# Exploration des bases Parcoursup 2018-2025

Objectif : construire une base unique (une ligne = une formation × une session) avec un identifiant de formation suivi dans le temps, une sélectivité et une filière harmonisées.

1. Importation des bases
2. Comparaison des colonnes entre années
3. Identifiant des formations
4. Sélectivité
5. Filières
6. Empilement des sessions
7. Nettoyage final et sauvegarde

> **État du notebook (novembre 2026).** C'est un notebook d'**exploration** : il garde la trace des vérifications qui justifient nos choix (identifiant, sélectivité, filières). Il produit `data/parcoursup_2018_2025.csv`, qui n'est pas versionné.
>
> **Suite prévue :** le code sera repris en fonctions dans des fichiers `.py`, importés par un notebook final. La base sera ensuite enrichie par des jointures (Île-de-France en priorité), puis analysée avec des graphiques. Le détail est dans le `README.md`.

## 1. Importation des bases

In [83]:
from pathlib import Path

import pandas as pd

# Identifiant de chaque session sur le portail open data du ministère
ids = {
    2018: "fr-esr-parcoursup-2018",
    2019: "fr-esr-parcoursup-2019",
    2020: "fr-esr-parcoursup_2020",
    2021: "fr-esr-parcoursup_2021",
    2022: "fr-esr-parcoursup_2022",
    2023: "fr-esr-parcoursup_2023",
    2024: "fr-esr-parcoursup_2024",
    2025: "fr-esr-parcoursup",
}

bases = {}
for annee, identifiant in ids.items():
    url = f"https://data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/catalog/datasets/{identifiant}/exports/csv?delimiter=%3B"
    bases[annee] = pd.read_csv(url, sep=";", low_memory=False, dtype={"dep": str, "cod_uai": str})
    print(annee, bases[annee].shape)

# low_memory : par défaut (True), pandas lit le fichier par morceaux pour économiser
# la mémoire et devine le type de chaque colonne morceau par morceau. Si deux morceaux
# n'ont pas le même type (des nombres puis du texte), la colonne mélange les types
# (pandas affiche alors un DtypeWarning). Avec low_memory=False, il lit toute la
# colonne avant de choisir un type unique.
# dtype : on impose le type texte à dep et cod_uai. Sinon, pandas pourrait lire "01"
# comme le nombre 1 (le zéro initial serait perdu), ou changer de type selon l'année
# à cause de "2A"/"2B" (Corse) et de la lettre finale des UAI. Le type est ainsi
# identique toutes les années au moment de l'empilement.

2018 (10697, 85)
2019 (11577, 92)
2020 (12760, 115)
2021 (13396, 118)
2022 (13644, 118)
2023 (13869, 118)
2024 (14079, 118)
2025 (14252, 118)


Le nombre de formations proposées augmente chaque année : il passe de 10 697 en 2018 à 14 252 en 2025. Le nombre de variables passe de 85 en 2018 à 118 en 2021, puis reste stable.

## 2. Comparaison des colonnes entre années

Nombre de colonnes communes à toutes les années, puis, pour chaque année, les colonnes de 2025 qui y sont absentes.

In [84]:
communes = set(bases[2018].columns)
for annee in bases:
    communes = communes & set(bases[annee].columns)
print(len(communes), "colonnes communes à toutes les années")

for annee in bases:
    absentes = set(bases[2025].columns) - set(bases[annee].columns)
    print(annee, len(absentes), sorted(absentes))

83 colonnes communes à toutes les années
2018 35 ['acc_datebac', 'acc_debutpp', 'acc_finpp', 'acc_tbf', 'cod_aff_form', 'composante_id_paysage', 'contrat_etab', 'detail_forma2', 'etablissement_id_paysage', 'lib_comp_voe_ins', 'lib_for_voe_ins', 'lib_grp1', 'lib_grp2', 'lib_grp3', 'list_com', 'part_acces_gen', 'part_acces_pro', 'part_acces_tec', 'pct_acc_datebac', 'pct_acc_debutpp', 'pct_acc_finpp', 'pct_tbf', 'prop_tot_at', 'prop_tot_bg', 'prop_tot_bg_brs', 'prop_tot_bp', 'prop_tot_bp_brs', 'prop_tot_bt', 'prop_tot_bt_brs', 'ran_grp1', 'ran_grp2', 'ran_grp3', 'select_form', 'taux_acces_ens', 'ville_etab']
2019 28 ['acc_tbf', 'cod_aff_form', 'composante_id_paysage', 'detail_forma2', 'etablissement_id_paysage', 'lib_comp_voe_ins', 'lib_for_voe_ins', 'lib_grp1', 'lib_grp2', 'lib_grp3', 'list_com', 'part_acces_gen', 'part_acces_pro', 'part_acces_tec', 'pct_tbf', 'prop_tot_at', 'prop_tot_bg', 'prop_tot_bg_brs', 'prop_tot_bp', 'prop_tot_bp_brs', 'prop_tot_bt', 'prop_tot_bt_brs', 'ran_grp1', 

Inversement, pour chaque année, les colonnes qui existent cette année-là mais plus en 2025.

In [85]:
for annee in bases:
    en_plus = set(bases[annee].columns) - set(bases[2025].columns)
    print(annee, sorted(en_plus))
# Certaines de ces colonnes ont peut-être été renommées plutôt que supprimées
# (par exemple taux_adm_psup en 2020-2021 et taux_acces_ens à partir de 2022).
# C'est une hypothèse, pas encore vérifiée.

2018 ['rang_der_max', 'regr_forma']
2019 ['rang_der_max', 'regr_forma']
2020 ['lib_grp4', 'lib_grp5', 'ran_grp4', 'ran_grp5', 'regr_forma', 'taux_adm_psup', 'taux_adm_psup_gen', 'taux_adm_psup_pro', 'taux_adm_psup_techno']
2021 ['taux_adm_psup', 'taux_adm_psup_gen', 'taux_adm_psup_pro', 'taux_adm_psup_techno']
2022 []
2023 []
2024 []
2025 []


## 3. Identifiant des formations

Hypothèse : le lien vers la fiche Parcoursup (`lien_form_psup`) contient l'identifiant de la formation (`g_ta_cod=...`). On le vérifie en 2025, seule année où l'on dispose aussi de `cod_aff_form`.

In [86]:
b25 = bases[2025]
code_lien = b25["lien_form_psup"].str.extract(r"g_ta_cod=(\d+)")[0]
print((code_lien.astype(float) == b25["cod_aff_form"]).mean())
# str.extract renvoie du texte : on le convertit en nombre pour comparer avec cod_aff_form.
# On passe par float plutôt que int, car un entier ne peut pas contenir de valeur
# manquante (NaN). Une comparaison avec NaN donne False : les liens vides comptent
# donc comme des écarts dans cette proportion.

0.9973337075498175


In [87]:
ecarts = b25[code_lien.astype(float) != b25["cod_aff_form"]]
print(len(ecarts), "écarts, dont", ecarts["lien_form_psup"].isna().sum(), "avec un lien vide")

38 écarts, dont 38 avec un lien vide


Les 38 écarts correspondent tous à des liens vides. Quand le lien est renseigné, le code extrait est toujours égal à `cod_aff_form`. Le lien contient donc bien l'identifiant de la formation, ce qui permet de récupérer le code pour 2018 et 2019, où `cod_aff_form` n'existe pas.

Création de la colonne `code` pour toutes les années : `cod_aff_form` à partir de 2020, le code extrait du lien en 2018 et 2019. On vérifie ensuite qu'il n'y a pas de doublon et que les codes sont stables entre 2019 et 2020.

In [88]:
for a in bases:
    if "cod_aff_form" in bases[a].columns:
        bases[a]["code"] = bases[a]["cod_aff_form"]
    else:
        bases[a]["code"] = bases[a]["lien_form_psup"].str.extract(r"g_ta_cod=(\d+)")[0].astype(float)
    print(a, "codes manquants :", bases[a]["code"].isna().sum(),
          "| doublons :", bases[a]["code"].dropna().duplicated().sum())
# Codes manquants : uniquement en 2018 et 2019, quand le lien est vide.
# Aucun doublon : chaque code désigne une seule formation dans une session.

codes_2019 = set(bases[2019]["code"].dropna().astype(int))
codes_2020 = set(bases[2020]["code"].astype(int))
print(len(codes_2019 & codes_2020) / len(codes_2019))
# Part des codes de 2019 que l'on retrouve en 2020 : 99,9 %. Les codes sont donc
# stables d'une année à l'autre et peuvent servir à suivre une formation dans le temps.

2018 codes manquants : 817 | doublons : 0
2019 codes manquants : 624 | doublons : 0
2020 codes manquants : 0 | doublons : 0
2021 codes manquants : 0 | doublons : 0
2022 codes manquants : 0 | doublons : 0
2023 codes manquants : 0 | doublons : 0
2024 codes manquants : 0 | doublons : 0
2025 codes manquants : 0 | doublons : 0
0.9994522048753766


Pour les formations appariées par leur code entre 2019 et 2020 : l'établissement et le libellé sont-ils les mêmes ?

In [89]:
colonnes = ["code", "cod_uai", "g_ea_lib_vx", "fil_lib_voe_acc"]
m_19_20 = bases[2019][colonnes].merge(bases[2020][colonnes], on="code", suffixes=("_2019", "_2020"))
print(len(m_19_20), "formations appariées")
print("même établissement :", (m_19_20["cod_uai_2019"] == m_19_20["cod_uai_2020"]).mean())
print("même libellé :", (m_19_20["fil_lib_voe_acc_2019"] == m_19_20["fil_lib_voe_acc_2020"]).mean())
# merge garde les formations présentes les deux années ; les colonnes qui existent
# des deux côtés reçoivent le suffixe _2019 ou _2020.

10947 formations appariées
même établissement : 0.9795377729058189
même libellé : 0.9530464967571024


In [90]:
diff = m_19_20[m_19_20["cod_uai_2019"] != m_19_20["cod_uai_2020"]]
print(len(diff))
print(diff[["g_ea_lib_vx_2019", "g_ea_lib_vx_2020"]].value_counts().head(15))
# Formations qui changent d'UAI entre 2019 et 2020 : on compare le nom de leur
# établissement les deux années (15 cas les plus fréquents).

224
g_ea_lib_vx_2019                                      g_ea_lib_vx_2020                                                                                    
Aix Marseille Université - site d'Aix-en-Provence     Aix Marseille Université - Site d'Aix-en-Provence                                                       57
Université Paris Diderot - Paris 7                    Université de Paris                                                                                     40
Université de Cergy-Pontoise                          Université de Cergy-Pontoise - CY SUP                                                                   27
Université Paris-Sud, campus d'Orsay                  Ecole Universitaire de premier cycle - Campus d'Orsay Université de Paris-Saclay (Orsay)                13
Université Paris Descartes Paris 5                    Université de Paris                                                                                     12
Aix-Marseille Université - Site de M

Les changements d'établissement correspondent à des fusions (Université de Paris, Paris-Saclay, CY), à des réorganisations de sites (Aix-Marseille) ou à des renommages. Ce sont les mêmes formations, rattachées à un établissement qui a changé d'identifiant.

In [91]:
diff_lib = m_19_20[m_19_20["fil_lib_voe_acc_2019"] != m_19_20["fil_lib_voe_acc_2020"]]
print(len(diff_lib))
print(diff_lib[["fil_lib_voe_acc_2019", "fil_lib_voe_acc_2020"]].value_counts().head(20))
# La plupart des 514 changements de libellé viennent d'une nouvelle façon de nommer
# les formations en 2020 (par exemple, "bac S" pour une école d'ingénieur devient
# "Formation d'ingénieur Bac + 5"), ou de corrections d'orthographe.
# Ce ne sont pas de nouvelles formations.

514
fil_lib_voe_acc_2019                                                                        fil_lib_voe_acc_2020                                                               
bac S                                                                                       Formation d'ingénieur Bac + 5                                                          148
bac STI2D                                                                                   Formation d'ingénieur Bac + 5                                                           58
Commerce                                                                                    Formation des écoles de commerce et de management Bac + 3                               35
Bac +1/+2                                                                                   Formation d'ingénieur Bac + 5                                                           29
Cycle Préparatoire Intégré - Spécialité Généraliste, BTP, Informatique, Systèmes embarqu

Formations sans lien en 2018-2019 (donc sans code), par filière.

In [92]:
for annee in [2018, 2019]:
    b = bases[annee]
    sans_lien = b["lien_form_psup"].isna()
    print(annee)
    print(sans_lien.groupby(b["fili"]).agg(["sum", "mean"]).sort_values("sum", ascending=False))
# sum : nombre de formations sans lien ; mean : part des formations de la filière sans lien.
# Les formations sans code sont surtout des "Autre formation" et des licences.
# En 2019, aucune PACES n'a de lien (mean = 1).

2018
                 sum      mean
fili                          
Autre formation  393  0.264825
Licence          294  0.120789
BTS              126  0.024706
DUT                3  0.003802
PACES              1  0.022727
CPGE               0  0.000000
2019
                 sum      mean
fili                          
Autre formation  251  0.146870
Licence          214  0.084887
BTS               79  0.015463
PACES             43  1.000000
EFTS              17  0.076923
IFSI              12  0.037037
DUT                5  0.006242
CPGE               3  0.003534


## 4. Sélectivité

In [93]:
for a in range(2020, 2025):
    avant = bases[a][["code", "select_form"]]
    apres = bases[a + 1][["code", "select_form"]]
    paire = avant.merge(apres, on="code", suffixes=("_avant", "_apres"))
    change = paire["select_form_avant"] != paire["select_form_apres"]
    print(a, "→", a + 1, ":", len(paire), "formations,", round(change.mean() * 100, 2), "% changent de statut")

# 100 % de changement entre 2020 et 2021 : ce n'est pas plausible. On soupçonne une
# différence dans la façon d'écrire les modalités en 2020. On regarde leurs valeurs.

2020 → 2021 : 12128 formations, 100.0 % changent de statut
2021 → 2022 : 12751 formations, 0.06 % changent de statut
2022 → 2023 : 13113 formations, 0.05 % changent de statut
2023 → 2024 : 13332 formations, 0.04 % changent de statut
2024 → 2025 : 13540 formations, 0.01 % changent de statut


In [94]:
for a in range(2020, 2026):
    print(a, bases[a]["select_form"].value_counts(dropna=False).to_dict())
# Confirmé : en 2020, les modalités sont écrites sans accent, et "non sélective" est
# tronqué en "non selec". Les effectifs sont proches d'une année à l'autre : le problème
# vient uniquement de l'écriture des modalités, pas d'un vrai changement de statut.

2020 {'formation selective': 10035, 'formation non selec': 2725}
2021 {'formation sélective': 10507, 'formation non sélective': 2889}
2022 {'formation sélective': 10616, 'formation non sélective': 3028}
2023 {'formation sélective': 10765, 'formation non sélective': 3104}
2024 {'formation sélective': 10891, 'formation non sélective': 3188}
2025 {'formation sélective': 11056, 'formation non sélective': 3196}


In [95]:
correspondance = {
    "formation selective": 1,
    "formation non selec": 0,
    "formation sélective": 1,
    "formation non sélective": 0,
}

for a in range(2020, 2026):
    bases[a]["selective"] = bases[a]["select_form"].map(correspondance)
    print(a, "non reconnues :", bases[a]["selective"].isna().sum())
# Nouvelle colonne selective (1 = sélective, 0 = non sélective), codée de la même
# façon toutes les années. map renvoie NaN pour une valeur absente du dictionnaire :
# 0 valeur non reconnue, donc le dictionnaire couvre toutes les modalités.

2020 non reconnues : 0
2021 non reconnues : 0
2022 non reconnues : 0
2023 non reconnues : 0
2024 non reconnues : 0
2025 non reconnues : 0


In [96]:
for a in range(2020, 2025):
    avant = bases[a][["code", "selective"]]
    apres = bases[a + 1][["code", "selective"]]
    paire = avant.merge(apres, on="code", suffixes=("_avant", "_apres"))
    change = paire["selective_avant"] != paire["selective_apres"]
    print(a, "→", a + 1, ":", len(paire), "formations,", round(change.mean() * 100, 2), "% changent de statut")
# Après harmonisation, moins de 0,1 % des formations changent de statut chaque année
# entre 2020 et 2025. On fait l'hypothèse que c'était aussi le cas en 2018-2019,
# ce qui permet de reporter le statut de 2020 sur ces deux années. On ne peut pas
# le vérifier (select_form n'existe pas avant 2020) : c'est une limite à mentionner.

2020 → 2021 : 12128 formations, 0.08 % changent de statut
2021 → 2022 : 12751 formations, 0.06 % changent de statut
2022 → 2023 : 13113 formations, 0.05 % changent de statut
2023 → 2024 : 13332 formations, 0.04 % changent de statut
2024 → 2025 : 13540 formations, 0.01 % changent de statut


Report du statut sélectif de 2020 sur 2018 et 2019, grâce au `code`.

In [97]:
# Série qui associe à chaque code son statut sélectif en 2020
reference = bases[2020].set_index("code")["selective"]

for a in [2018, 2019]:
    bases[a]["selective"] = bases[a]["code"].map(reference)
    print(a, "statut inconnu :", bases[a]["selective"].isna().sum(), "sur", len(bases[a]))
# Statut inconnu : formation sans code (lien vide) ou absente en 2020.
# On complétera une partie de ces cas grâce à la filière (fin de la partie 5).

2018 statut inconnu : 1132 sur 10697
2019 statut inconnu : 630 sur 11577


## 5. Filières

In [98]:
for annee in bases:
    print(annee, sorted(bases[annee]["fili"].dropna().unique()))
# Les modalités de fili changent selon les années : DUT devient BUT en 2021, PACES devient
# PASS et LAS en 2020, et les écoles d'ingénieurs et de commerce apparaissent en 2020.

2018 ['Autre formation', 'BTS', 'CPGE', 'DUT', 'Licence', 'PACES']
2019 ['Autre formation', 'BTS', 'CPGE', 'DUT', 'EFTS', 'IFSI', 'Licence', 'PACES']
2020 ['Autre formation', 'BTS', 'CPGE', 'DUT', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']
2021 ['Autre formation', 'BTS', 'BUT', 'CPGE', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']
2022 ['Autre formation', 'BTS', 'BUT', 'CPGE', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']
2023 ['Autre formation', 'BTS', 'BUT', 'CPGE', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']
2024 ['Autre formation', 'BTS', 'BUT', 'CPGE', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']
2025 ['Autre formation', 'BTS', 'BUT', 'CPGE', 'EFTS', "Ecole d'Ingénieur", 'Ecole de Commerce', 'IFSI', 'Licence', 'Licence_Las', 'PASS']


Nomenclature commune. Le report de 2020 ne concerne que les formations « Autre formation » de 2018-2019 (écoles d'ingénieurs et de commerce isolées à partir de 2020).

In [99]:
nomenclature = {
    "CPGE": "CPGE",
    "BTS": "BTS",
    "DUT": "IUT",
    "BUT": "IUT",
    "Licence": "Licence",
    "Licence_Las": "LAS",
    "PACES": "Santé",
    "PASS": "Santé",
    "Ecole d'Ingénieur": "Ecole d'ingénieur",
    "Ecole de Commerce": "Ecole de commerce",
    "EFTS": "Sanitaire et social",
    "IFSI": "Sanitaire et social",
    "Autre formation": "Autre",
}

for a in bases:
    bases[a]["filiere"] = bases[a]["fili"].map(nomenclature)
    print(a, "non reconnues :", bases[a]["filiere"].isna().sum())

reference_fil = bases[2020].set_index("code")["filiere"]

for a in [2018, 2019]:
    b = bases[a]
    autre = b["fili"] == "Autre formation"
    b.loc[autre, "filiere"] = b.loc[autre, "code"].map(reference_fil).fillna("Autre")

print(pd.crosstab(bases[2019]["fili"], bases[2019]["filiere"]))

2018 non reconnues : 0
2019 non reconnues : 0
2020 non reconnues : 0
2021 non reconnues : 0
2022 non reconnues : 0
2023 non reconnues : 0
2024 non reconnues : 0
2025 non reconnues : 0
filiere          Autre   BTS  CPGE  Ecole d'ingénieur  Ecole de commerce  IUT  \
fili                                                                            
Autre formation   1349     0     0                313                 47    0   
BTS                  0  5109     0                  0                  0    0   
CPGE                 0     0   849                  0                  0    0   
DUT                  0     0     0                  0                  0  801   
EFTS                 0     0     0                  0                  0    0   
IFSI                 0     0     0                  0                  0    0   
Licence              0     0     0                  0                  0    0   
PACES                0     0     0                  0                  0    0   

filie

Les LAS de 2020 existaient-elles déjà en 2019 sous le même code ?

In [100]:
las_2020 = set(bases[2020].loc[bases[2020]["fili"] == "Licence_Las", "code"].astype(int))
codes_2019 = set(bases[2019]["code"].dropna().astype(int))
print(len(las_2020), "LAS en 2020")
print(len(las_2020 & codes_2019) / len(las_2020), "déjà présentes en 2019")
# Seulement 5 % des LAS de 2020 existaient déjà en 2019 : ce sont presque toutes de
# nouvelles formations, créées par la réforme des études de santé de 2020.

457 LAS en 2020
0.05032822757111598 déjà présentes en 2019


Compléter la sélectivité de 2018-2019 grâce à la filière. On regarde d'abord, entre 2020 et 2025, la part de formations sélectives dans chaque filière.

In [101]:
recentes = pd.concat([bases[a] for a in range(2020, 2026)], ignore_index=True)
print(pd.crosstab(recentes["filiere"], recentes["selective"], normalize="index").round(3))
# ignore_index=True : on renumérote les lignes de 0 à n. Sans ça, chaque année garde
# ses propres numéros (0, 1, 2...), on a des doublons d'index et crosstab échoue.
# normalize="index" : chaque ligne est divisée par son total, on lit donc des parts.
# Entre 2020 et 2025, toutes les formations sont sélectives en BTS, CPGE, IUT, écoles,
# sanitaire et social et "Autre", et aucune ne l'est en santé (PASS).
# Les licences sont mixtes : on ne peut pas deviner leur statut à partir de la filière.

selective                0      1
filiere                          
Autre                0.000  1.000
BTS                  0.000  1.000
CPGE                 0.000  1.000
Ecole d'ingénieur    0.000  1.000
Ecole de commerce    0.000  1.000
IUT                  0.000  1.000
LAS                  0.975  0.025
Licence              0.789  0.211
Sanitaire et social  0.000  1.000
Santé                1.000  0.000


In [102]:
# Statut de chaque filière quand il est le même pour toutes ses formations (tableau ci-dessus)
statut_filiere = {
    "BTS": 1,
    "CPGE": 1,
    "IUT": 1,
    "Ecole d'ingénieur": 1,
    "Ecole de commerce": 1,
    "Sanitaire et social": 1,
    "Autre": 1,
    "Santé": 0,
}

for a in [2018, 2019]:
    b = bases[a]
    inconnu = b["selective"].isna()
    b.loc[inconnu, "selective"] = b.loc[inconnu, "filiere"].map(statut_filiere)
    print(a, "statut encore inconnu :", b["selective"].isna().sum(), "dont licences :",
          (b["selective"].isna() & (b["filiere"] == "Licence")).sum())
# On ne remplit que les statuts manquants (inconnu), sans toucher à ceux venus de 2020.
# Il ne reste que des licences : leur statut reste manquant.

2018 statut encore inconnu : 420 dont licences : 420
2019 statut encore inconnu : 214 dont licences : 214


## 6. Empilement des sessions

In [103]:
base = pd.concat(bases.values(), ignore_index=True)

# Indicateur : formation présente les 8 années
nb_annees = base.dropna(subset=["code"]).groupby("code")["session"].nunique()
base["toutes_annees"] = base["code"].map(nb_annees).eq(8)

print(base.shape)
print(base.groupby("session").size())
print("formations présentes les 8 années :", base.loc[base["toutes_annees"], "code"].nunique())

(104274, 132)
session
2018    10697
2019    11577
2020    12760
2021    13396
2022    13644
2023    13869
2024    14079
2025    14252
dtype: int64
formations présentes les 8 années : 8566


Colonnes renseignées selon la session (1 = au moins une valeur non vide cette année-là). Seules les colonnes qui ne sont pas renseignées toutes les années sont affichées.

In [104]:
presence = base.notna().groupby(base["session"]).any().T
pd.set_option("display.max_rows", None)
print(presence[~presence.all(axis=1)].astype(int).to_string())

session                   2018  2019  2020  2021  2022  2023  2024  2025
regr_forma                   1     1     1     0     0     0     0     0
rang_der_max                 1     1     0     0     0     0     0     0
contrat_etab                 0     1     1     1     1     1     1     1
acc_debutpp                  0     1     1     1     1     1     1     1
acc_datebac                  0     1     1     1     1     1     1     1
acc_finpp                    0     1     1     1     1     1     1     1
pct_acc_debutpp              0     1     1     1     1     1     1     1
pct_acc_datebac              0     1     1     1     1     1     1     1
pct_acc_finpp                0     1     1     1     1     1     1     1
select_form                  0     0     1     1     1     1     1     1
lib_comp_voe_ins             0     0     1     1     1     1     1     1
prop_tot_bg                  0     0     1     1     1     1     1     1
prop_tot_bg_brs              0     0     1     1   

## 7. Nettoyage final et sauvegarde

En 2018, 189 lignes ont la mention « identifiant obsolète » à la place de l'UAI (dans le fichier source). On la remplace par une valeur manquante.

In [105]:
base.loc[base["cod_uai"] == "identifiant obsolète", "cod_uai"] = pd.NA
print("mentions restantes :", (base["cod_uai"] == "identifiant obsolète").sum())

# Contrôles
print("filière manquante :", base["filiere"].isna().sum())
print("sélectivité manquante :", base["selective"].isna().sum())

Path("../data").mkdir(exist_ok=True)
base.to_csv("../data/parcoursup_2018_2025.csv", sep=";", index=False)

mentions restantes : 0
filière manquante : 0
sélectivité manquante : 634
